In [8]:
import os
import cdsapi

# Local data folder for ORAS5 outputs
DATA_DIR = '../../data/ORAS5'
os.makedirs(DATA_DIR, exist_ok=True)

dataset = "reanalysis-oras5"
request = {
    "product_type": ["operational"],
    "vertical_resolution": "single_level",
    "variable": [
        "depth_of_20_c_isotherm",
        "ocean_heat_content_for_the_upper_300m",
        "sea_surface_height"
    ],
    "year": ["2024"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    # Same grid and region as the ERA5 dataset (El Nino Pacific box).
    # ORAS5 does not wrap longitudes across the dateline like ERA5, so the
    # eastern bound -80 is expressed in 0-360 convention as 280.
    "grid": [2.0, 2.0],
    "area": [30, 120, -30, 280],  # N, W, S, E
    "data_format": "netcdf"
}

client = cdsapi.Client()
target = os.path.join(DATA_DIR, 'oras5_2024.zip')
client.retrieve(dataset, request, target)
print(f"Downloaded to {target}")


2026-06-25 22:03:03,611 INFO [2026-04-30T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-oras5-timeseries?tab=overview).
2026-06-25 22:03:03,612 INFO Request ID is 1e20138c-50bf-4b36-af06-02ac7feb363c
2026-06-25 22:03:05,691 INFO status has been updated to accepted
2026-06-25 22:03:30,815 INFO status has been updated to failed


2026-06-25 22:03:03,611 INFO [2026-04-30T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-oras5-timeseries?tab=overview).
2026-06-25 22:03:03,612 INFO Request ID is 1e20138c-50bf-4b36-af06-02ac7feb363c
2026-06-25 22:03:05,691 INFO status has been updated to accepted
2026-06-25 22:03:30,815 INFO status has been updated to failed


HTTPError: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/jobs/1e20138c-50bf-4b36-af06-02ac7feb363c/results
The job has failed
Area selection resulted in a dataset with zero length dimension for: y.
Please ensure that your area selection covers at least one point in the data.
The job failed with: InvalidRequest

In [ ]:
import os
import glob
import json
import zipfile
import xarray as xr

DATA_DIR = '../../data/ORAS5'
zip_path = os.path.join(DATA_DIR, 'oras5_2024.zip')

# ORAS5 downloads arrive as a zip of NetCDF files (one per variable/month)
extract_dir = os.path.join(DATA_DIR, 'nc')
os.makedirs(extract_dir, exist_ok=True)
if zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    nc_files = sorted(glob.glob(os.path.join(extract_dir, '*.nc')))
else:
    # Single NetCDF file was returned
    nc_files = [zip_path]

print(f"Found {len(nc_files)} NetCDF file(s)")

# Merge all variables/months into one dataset
ds = xr.open_mfdataset(nc_files, combine='by_coords')

# Save metadata
metadata = {
    'dimensions': dict(ds.sizes),
    'variables': {},
    'global_attributes': dict(ds.attrs)
}
for var in ds.variables:
    metadata['variables'][var] = {
        'dims': list(ds[var].dims),
        'shape': list(ds[var].shape),
        'dtype': str(ds[var].dtype),
        'attributes': dict(ds[var].attrs)
    }

meta_path = os.path.join(DATA_DIR, 'oras5_2024_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"Metadata saved to {meta_path}")

# Convert to DataFrame and save as CSV
csv_path = os.path.join(DATA_DIR, 'oras5_2024.csv')
df = ds.to_dataframe().reset_index()
df.to_csv(csv_path, index=False)
print(f"CSV saved to {csv_path} ({len(df)} rows)")

ds.close()


In [ ]:
import os
import pickle
import mimetypes
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

SCOPES = ['https://www.googleapis.com/auth/drive']
TOKEN_PATH = '../../token.pickle'
CLIENT_SECRETS = '../../client_secrets.json'

# Parent Drive folder under which all data folders are uploaded
DRIVE_PARENT_ID = '1zLJgkYkrM1LRgtl7v5sfAQ4aits9zfUq'

DATA_DIR = '../../data/ORAS5'

# Only upload the CSV and the metadata (two files)
FILES_TO_UPLOAD = [
    os.path.join(DATA_DIR, 'oras5_2024.csv'),
    os.path.join(DATA_DIR, 'oras5_2024_metadata.json'),
]


def get_drive_service():
    creds = None
    if os.path.exists(TOKEN_PATH):
        with open(TOKEN_PATH, 'rb') as f:
            creds = pickle.load(f)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRETS, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(TOKEN_PATH, 'wb') as f:
            pickle.dump(creds, f)
    return build('drive', 'v3', credentials=creds)


def get_or_create_folder(service, name, parent_id):
    """Return the Drive folder id for `name` under `parent_id`, creating it if needed."""
    query = (
        f"name = '{name}' and mimeType = 'application/vnd.google-apps.folder' "
        f"and '{parent_id}' in parents and trashed = false"
    )
    response = service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
    files = response.get('files', [])
    if files:
        return files[0]['id']
    folder = service.files().create(
        body={'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]},
        fields='id'
    ).execute()
    return folder['id']


service = get_drive_service()

# Create (or reuse) the ORAS folder to hold the CSV and metadata
oras_folder_id = get_or_create_folder(service, 'ORAS', DRIVE_PARENT_ID)
print(f"Folder: ORAS (Drive id: {oras_folder_id})")

for path in FILES_TO_UPLOAD:
    name = os.path.basename(path)
    mimetype = mimetypes.guess_type(path)[0] or 'application/octet-stream'
    media = MediaFileUpload(path, mimetype=mimetype, resumable=True)
    result = service.files().create(
        body={'name': name, 'parents': [oras_folder_id]},
        media_body=media,
        fields='id, name'
    ).execute()
    print(f"  Uploaded: {result['name']} (id: {result['id']})")

print("Done.")


# ORAS5 Anomaly Computation & Z-Score Normalization
Same methodology as ERA5:
- **Anomaly baseline**: 1980-01-01 to 2010-12-31 (monthly climatology per lat/lon)
- **Z-score normalization**: training period 1980-01-01 to 2018-12-31, applied to anomaly columns only

In [1]:
import pandas as pd
import numpy as np

# ── Load ORAS5 combined CSV ──
oras_path = '/Users/raman/Elnino/data/ORAS5/oras5_combined_1980_2026.csv'
df_oras = pd.read_csv(oras_path)
df_oras['time'] = pd.to_datetime(df_oras['time'])
df_oras['month'] = df_oras['time'].dt.month
df_oras['latitude'] = df_oras['latitude'].round(4)
df_oras['longitude'] = df_oras['longitude'].round(4)

print(f'Loaded: {oras_path}')
print(f'Shape: {df_oras.shape}')
print(f'Columns: {df_oras.columns.tolist()}')
print(f'Time range: {df_oras["time"].min()} to {df_oras["time"].max()}')
df_oras.head(3)

Loaded: /Users/raman/Elnino/data/ORAS5/oras5_combined_1980_2026.csv
Shape: (1398627, 10)
Columns: ['time', 'latitude', 'longitude', 'so20chgt', 'sohtc300', 'sossheig', 'lsm', 'land_ocean', 'land_ocean_flag', 'month']
Time range: 1980-01-01 00:00:00 to 2026-05-01 00:00:00


,time,latitude,longitude,so20chgt,sohtc300,sossheig,lsm,land_ocean,land_ocean_flag,month
0,1980-01-01,-30.0,120.0,NaN,NaN,NaN,1.000000,land,1,1
1,1980-01-01,-30.0,122.0,NaN,NaN,NaN,0.899545,land,1,1
2,1980-01-01,-30.0,124.0,NaN,NaN,NaN,1.000000,land,1,1


In [2]:
# ── Anomaly Computation (baseline: 1980-2010) ──
BASELINE_START = '1980-01-01'
BASELINE_END   = '2010-12-31'
CLIM_VARS_ORAS = ['so20chgt', 'sohtc300', 'sossheig']

baseline_oras = df_oras[(df_oras['time'] >= BASELINE_START) & (df_oras['time'] <= BASELINE_END)]

clim_mean_oras = (
    baseline_oras
    .groupby(['latitude', 'longitude', 'month'])[CLIM_VARS_ORAS]
    .mean()
    .rename(columns={v: f'{v}_clim' for v in CLIM_VARS_ORAS})
    .reset_index()
)

df_oras = df_oras.merge(clim_mean_oras, on=['latitude', 'longitude', 'month'], how='left')
for v in CLIM_VARS_ORAS:
    df_oras[f'{v}_anom'] = df_oras[v] - df_oras[f'{v}_clim']

df_oras.drop(columns=['month'] + [f'{v}_clim' for v in CLIM_VARS_ORAS], inplace=True)
anom_cols_oras = [f'{v}_anom' for v in CLIM_VARS_ORAS]

print('Anomaly columns created:', anom_cols_oras)
print('Non-NaN anomaly counts:')
for c in anom_cols_oras:
    print(f'  {c}: {df_oras[c].notna().sum()}')
df_oras[['time', 'latitude', 'longitude'] + CLIM_VARS_ORAS + anom_cols_oras].dropna().head(5)

Anomaly columns created: ['so20chgt_anom', 'sohtc300_anom', 'sossheig_anom']
Non-NaN anomaly counts:
  so20chgt_anom: 1276087
  sohtc300_anom: 1276087
  sossheig_anom: 1276087


,time,latitude,longitude,so20chgt,sohtc300,sossheig,so20chgt_anom,sohtc300_anom,sossheig_anom
17,1980-01-01,-30.0,154.0,126.044464,2.490480e+10,0.508364,-7.263901,-4.362329e+08,-0.045953
18,1980-01-01,-30.0,156.0,83.309395,2.341348e+10,0.455537,-40.115628,-1.259553e+09,-0.147461
19,1980-01-01,-30.0,158.0,100.968727,2.414445e+10,0.550122,-1.405746,5.360600e+07,0.002799
20,1980-01-01,-30.0,160.0,67.495941,2.338174e+10,0.502701,-12.814394,-3.335736e+08,-0.018495
21,1980-01-01,-30.0,162.0,62.692909,2.307102e+10,0.484665,-5.216814,-2.168372e+08,-0.006937


In [3]:
# ── Z-score Normalization (training period: 1980-2018) ──
TRAIN_START = '1980-01-01'
TRAIN_END   = '2018-12-31'

norm_cols_oras = anom_cols_oras
train_oras = df_oras[(df_oras['time'] >= TRAIN_START) & (df_oras['time'] <= TRAIN_END)]

mu_oras = train_oras[norm_cols_oras].mean()
sigma_oras = train_oras[norm_cols_oras].std(ddof=0).replace(0, np.nan)

for c in norm_cols_oras:
    df_oras[f'{c}_z'] = (df_oras[c] - mu_oras[c]) / sigma_oras[c]

z_cols_oras = [f'{c}_z' for c in norm_cols_oras]

# Save normalization stats
pd.DataFrame({'mean': mu_oras, 'std': sigma_oras}).to_csv(
    '/Users/raman/Elnino/data/ORAS5/oras5_norm_stats_1980_2018.csv'
)

print('Z-score columns created:', z_cols_oras)
print('\nNormalization stats (training 1980-2018):')
print(pd.DataFrame({'mean': mu_oras, 'std': sigma_oras}))
print('\nSample z-scored ocean rows:')
df_oras[['time', 'latitude', 'longitude'] + anom_cols_oras + z_cols_oras].dropna().head(5)

Z-score columns created: ['so20chgt_anom_z', 'sohtc300_anom_z', 'sossheig_anom_z']

Normalization stats (training 1980-2018):
                       mean           std
so20chgt_anom  9.277411e-01  1.596041e+01
sohtc300_anom  4.250766e+07  8.001026e+08
sossheig_anom  8.686595e-03  5.897240e-02

Sample z-scored ocean rows:


,time,latitude,longitude,so20chgt_anom,sohtc300_anom,sossheig_anom,so20chgt_anom_z,sohtc300_anom_z,sossheig_anom_z
17,1980-01-01,-30.0,154.0,-7.263901,-4.362329e+08,-0.045953,-0.513247,-0.598349,-0.926532
18,1980-01-01,-30.0,156.0,-40.115628,-1.259553e+09,-0.147461,-2.571573,-1.627368,-2.647808
19,1980-01-01,-30.0,158.0,-1.405746,5.360600e+07,0.002799,-0.146205,0.013871,-0.099842
20,1980-01-01,-30.0,160.0,-12.814394,-3.335736e+08,-0.018495,-0.861014,-0.470041,-0.460913
21,1980-01-01,-30.0,162.0,-5.216814,-2.168372e+08,-0.006937,-0.384987,-0.324140,-0.264937


In [4]:
# ── Export updated ORAS5 CSV ──
cols_to_export = (
    ['time', 'latitude', 'longitude']
    + CLIM_VARS_ORAS
    + anom_cols_oras
    + z_cols_oras
    + ['lsm', 'land_ocean', 'land_ocean_flag']
)
export_path = '/Users/raman/Elnino/data/ORAS5/oras5_with_original_anomaly_zscore.csv'
df_oras[cols_to_export].to_csv(export_path, index=False)
print(f'Exported to: {export_path}')
print(f'Shape: {df_oras[cols_to_export].shape}')

Exported to: /Users/raman/Elnino/data/ORAS5/oras5_with_original_anomaly_zscore.csv
Shape: (1398627, 15)


In [5]:
# ── Validation ──
print('='*72)
print('VALIDATION CHECKS')
print('='*72)

# 1. Anomalies should be ~0 over the baseline period for ocean points
ocean_mask = df_oras['land_ocean_flag'] == 0
baseline_mask = (df_oras['time'] >= BASELINE_START) & (df_oras['time'] <= BASELINE_END)

print('\n1. Mean anomaly over baseline (1980-2010) for OCEAN points (should be ≈ 0):')
for c in anom_cols_oras:
    m = df_oras.loc[ocean_mask & baseline_mask, c].mean()
    print(f'   {c}: {m:.6f}')

# 2. Z-scores: training-period mean ≈ 0 and std ≈ 1 for ocean points
train_mask = (df_oras['time'] >= TRAIN_START) & (df_oras['time'] <= TRAIN_END)
print('\n2. Z-score stats over training period (1980-2018) for OCEAN points:')
print('   (mean should be ≈ 0, std should be ≈ 1)')
for c in z_cols_oras:
    vals = df_oras.loc[ocean_mask & train_mask, c].dropna()
    print(f'   {c}: mean={vals.mean():.6f}, std={vals.std(ddof=0):.6f}')

# 3. Land points should remain NaN in anomaly and z-score columns
land_mask = df_oras['land_ocean_flag'] == 1
print('\n3. Land points should be all NaN in anomaly/z-score columns:')
for c in anom_cols_oras + z_cols_oras:
    n_nonnull = df_oras.loc[land_mask, c].notna().sum()
    status = '✓ all NaN' if n_nonnull == 0 else f'✗ {n_nonnull} non-null!'
    print(f'   {c}: {status}')

# 4. No NaN in ocean anomaly/z-score
print('\n4. Ocean points should have NO NaN in anomaly/z-score columns:')
for c in anom_cols_oras + z_cols_oras:
    n_null = df_oras.loc[ocean_mask, c].isna().sum()
    status = '✓ no NaN' if n_null == 0 else f'✗ {n_null} NaN!'
    print(f'   {c}: {status}')

# 5. Row counts
print(f'\n5. Total rows: {len(df_oras)}')
print(f'   Ocean: {ocean_mask.sum()}, Land: {land_mask.sum()}')

VALIDATION CHECKS

1. Mean anomaly over baseline (1980-2010) for OCEAN points (should be ≈ 0):
   so20chgt_anom: 0.000000
   sohtc300_anom: -0.000000
   sossheig_anom: -0.000000

2. Z-score stats over training period (1980-2018) for OCEAN points:
   (mean should be ≈ 0, std should be ≈ 1)
   so20chgt_anom_z: mean=0.000000, std=1.000000
   sohtc300_anom_z: mean=0.000000, std=1.000000
   sossheig_anom_z: mean=-0.000000, std=1.000000

3. Land points should be all NaN in anomaly/z-score columns:
   so20chgt_anom: ✓ all NaN
   sohtc300_anom: ✓ all NaN
   sossheig_anom: ✓ all NaN
   so20chgt_anom_z: ✓ all NaN
   sohtc300_anom_z: ✓ all NaN
   sossheig_anom_z: ✓ all NaN

4. Ocean points should have NO NaN in anomaly/z-score columns:
   so20chgt_anom: ✓ no NaN
   sohtc300_anom: ✓ no NaN
   sossheig_anom: ✓ no NaN
   so20chgt_anom_z: ✓ no NaN
   sohtc300_anom_z: ✓ no NaN
   sossheig_anom_z: ✓ no NaN

5. Total rows: 1398627
   Ocean: 1276087, Land: 122540
